### Section 4: Random Sample Generation:
The function generate_subset_sum_case generates an instance of the multiset, while the function save_uniform_test_cases utilizes the former to generate multiple random test cases of a requested set size and number of cases to be generated, it writes those cases in a unified format to a txt file.

This random sample generator will be mainly used for performance testing of both the Brute Force and Heuristic algorithms.

In [23]:
import random
from pathlib import Path 

def generate_subset_sum_case(T, n, method="bounded"):
    """
    Generates a multiset of n random integers.
    Methods:
      - "bounded": Traditional style where elements are <= T (Easier to find exact matches).
      - "wild": Unrestricted values where elements can be much larger than T (Triggers heavy optimization/worst-case).
    """
    if n == 0:
        return []
        
    if method == "bounded":
        lower_bound = 1 if T >= 1 else T
        multiset = [random.randint(lower_bound, T) for _ in range(n)]
    else:
        # Wild Generation: Elements are chosen completely independent of T
        # Using a wide range (e.g., 1 to 1000) makes exact subset sums much harder to come by
        multiset = [random.randint(1, 1000) for _ in range(n)]
        
    return multiset

def save_uniform_test_cases(filename, numCases, caseSize, min_T, max_T):
    """
    Generates numCases test cases, each precisely of size caseSize.
     Splits the generation 50/50 between bounded and wild instances
    to test both average and difficult optimization states.
    """
    # Ensure the parent directory exists before writing to prevent a FileNotFoundError
    Path(filename).parent.mkdir(parents=True, exist_ok=True)

    with open(filename, 'w') as f:
        f.write(f"{numCases}\n")
        
        half_cases = numCases // 2
        
        for i in range(numCases):
            # Generate a random target sum T for this specific case
            T = random.randint(min_T, max_T)
            
            # Determine generation strategy (First half bounded, second half wild)
            strategy = "bounded" if i < half_cases else "wild"
            
            # Generate the multiset using the fixed caseSize
            multiset = generate_subset_sum_case(T, caseSize, method=strategy)
            
            # Unified Format:
            # Line 1: T
            # Line 2: space-separated elements
            f.write(f"{T}\n")
            f.write(" ".join(map(str, multiset)) + "\n")

### Generating uniform tests for Brute Force and Heuristic Algorithms:

In [24]:
# --- Execution Script ---

# Total instances per size file
num_cases_to_generate = 30

# Bounds for the random target sum T
minimum_target = 20
maximum_target = 300

# Generating 30 files, corresponding to sizes 0 up to 29
for i in range(0, 30): 
    output_filename = f"uniform_test_size{i}.txt"
    target_dir = Path("../tests/uniform_tests")
    file_path = target_dir / output_filename
    
    save_uniform_test_cases(
        filename=file_path,
        numCases=num_cases_to_generate,
        caseSize=i,
        min_T=minimum_target,
        max_T=maximum_target
    )

print(f"Generated {num_cases_to_generate} test cases per file (for sizes 0 to {i}) inside '../tests/uniform_tests'.")

Generated 30 test cases per file (for sizes 0 to 29) inside '../tests/uniform_tests'.


### Section 5: Algorithm Implementations

### Brute Force Algorithm Implementation:

The following code implements the brute force algorithm for finding the subset that adds up to the sum

In [25]:
from itertools import combinations
def ExhaustiveAlgorithm(numbers, target):

    best_subset = []
    best_sum = 0

    n = len(numbers)

    for r in range(n + 1):

        for subset in combinations(numbers, r):

            s = sum(subset)

            if best_sum < s <= target:
                best_sum = s
                best_subset = list(subset)

            if best_sum == target:
                return best_subset, best_sum

    return best_subset, best_sum

### Heuristic Algorithm Implementation:

In [26]:
import random
def rgli_subset_sum(numbers, target, max_trials=40, seed=None):
    """
    Randomized Greedy with Local Improvement (RGLI)
    for the optimization version of Subset Sum.

    Returns:
        best_subset, best_sum
    """
    if seed is not None:
        random.seed(seed)

    best_subset = []
    best_sum = 0

    for _ in range(max_trials):
        # Phase 1: randomized greedy
        shuffled = numbers[:]
        random.shuffle(shuffled)

        current_subset = []
        current_sum = 0

        for a in shuffled:
            if current_sum + a <= target:
                current_subset.append(a)
                current_sum += a

        # Phase 2: local improvement
        improved = True

        while improved:
            improved = False

            selected = current_subset[:]
            random.shuffle(selected)
            unselected = numbers[:]

            # remove selected items from unselected
            for x in selected:
                unselected.remove(x)

            for chosen in selected:
                remaining_capacity = target - (current_sum - chosen)

                # find largest unselected item that fits after removing chosen
                candidates = [x for x in unselected if x <= remaining_capacity]

                if candidates:
                    replacement = max(candidates)

                    if replacement > chosen:
                        current_subset.remove(chosen)
                        current_subset.append(replacement)

                        current_sum = current_sum - chosen + replacement

                        improved = True
                        break

        # update global best
        if current_sum > best_sum:
            best_sum = current_sum
            best_subset = current_subset[:]

        if best_sum == target:
            break

    return best_subset, best_sum


### 5.1/5.2: Initial Testing of the Brute Force and Heuristic Algorithms:

First, we generate 15 random tests with varying sizes (1-18) for quick smoke testing.

In [ ]:
# Generate 15 random tests with varying sizes (1-18) in one file
output_filename = "../tests/initial_general_tests.txt"

with open(output_filename, 'w') as f:
    # Write total number of tests
    f.write("15\n")
    
    # Generate 15 tests with random sizes from 1 to 18
    for i in range(15):
        # Random size between 1 and 18
        test_size = random.randint(1, 18)
        
        # Random target sum between 20 and 300
        target_sum = random.randint(20, 300)
        
        # Generate multiset using the existing function
        multiset = generate_subset_sum_case(target_sum, test_size)
        
        # Write in format: size, target, elements
        f.write(f"{test_size}\n")
        f.write(f"{target_sum}\n")
        f.write(" ".join(map(str, multiset)) + "\n")

print("Generated 15 mixed-size tests in initial_general_tests.txt")

The following code implements initial testing for both algorithms, the Brute Force and the Heuristic, it uses random sets of arbitrary sizes and also checks the error offset of the hearistic solution.

In [6]:
import collections #NOTE: for initial testing 
from pathlib import Path

def run_all_file_verification(input_filename):
    """
    Iterates over all instances in a mixed test file sequentially using a 
    strict 3-line parsing architecture (Size -> Target -> Multiset).
    Runs the algorithm block and performs a 2-level verification check.
    
    Parameters:
        input_filename (str/Path): Path to the mixed test instances file.
    """
    with open(input_filename, 'r') as infile:
        # Load all non-empty lines cleanly
        lines = [line.strip() for line in infile if line.strip()]
    
    if not lines:
        print(f"Error: '{input_filename}' is empty.")
        return

    num_cases = int(lines[0])
    line_index = 1

    print(f"====================================================")
    print(f" STARTING ALL-INSTANCE EXPERIMENTAL VERIFICATION")
    print(f" Processing {num_cases} total instances from file")
    print(f"====================================================\n")

    success_count = 0

    for case_num in range(1, num_cases + 1):
        if line_index >= len(lines):
            break
            
        try:
            # 1. Parse current instance using the exact 3-line block structure
            expected_size = int(lines[line_index])
            T = int(lines[line_index + 1])
            line_index += 2  # Move past Size and Target lines
            
            multiset = []
            if expected_size > 0:
                multiset = list(map(int, lines[line_index].split()))
                line_index += 1  # Move past the Multiset line
                
                # Sanity check to make sure the file row matches the specified size
                if len(multiset) != expected_size:
                    raise ValueError(
                        f"Parser mismatch: File stated size {expected_size}, "
                        f"but parsed {len(multiset)} elements."
                    )
            else:
                # If expected_size is 0, there is no multiset row to read in the file
                multiset = []

            print(f"[Instance {case_num}/{num_cases}] Target T = {T} | Multiset Size = {len(multiset)}")
            
            # Ensure your algorithms assign their list results to these names:
            # Both algorithms now return (best_subset, best_sum)
            bf_result = ExhaustiveAlgorithm(multiset, T)[0]
            heur_result = rgli_subset_sum(multiset, T)[0]
            
            # 2. Level 1 Verification: Multiset & Duplicate Frequency Mapping
            def check_level_1_frequencies(subset, parent_set, name):
                if subset == ["NOT FOUND"]:
                    return
                parent_counts = collections.Counter(parent_set)
                sub_counts = collections.Counter(subset)
                for item, count in sub_counts.items():
                    if item not in parent_counts:
                        raise ValueError(
                            f"Level 1 Failure ({name}): Element {item} inside the "
                            f"returned subset does not exist in the original multiset."
                        )
                    if count > parent_counts[item]:
                        raise ValueError(
                            f"Level 1 Failure ({name}): Element {item} appears {count} times "
                            f"in the subset, but only {parent_counts[item]} times in the original multiset."
                        )

            # Run Level 1 checks
            check_level_1_frequencies(bf_result, multiset, "Brute Force")
            check_level_1_frequencies(heur_result, multiset, "Heuristic")

            # 3. Level 2 Verification: Optimization Summation Check
            bf_sum = sum(bf_result) if bf_result != ["NOT FOUND"] else 0
            heur_sum = sum(heur_result) if heur_result != ["NOT FOUND"] else 0
            
            bf_diff = abs(T - bf_sum)
            heur_diff = abs(T - heur_sum)

            # Proximity metrics function to prevent code repetition
            def print_proximity_report(name, current_sum, diff):
                accuracy_pct = (1 - (diff / T)) * 100 if T != 0 else 100.0
                print(f"  ⚠️  {name} did not find exact target match.")
                print(f"      ↳ Sub-optimal Sum: {current_sum} (Target: {T})")
                print(f"      ↳ Closeness: Off by {diff} units ({accuracy_pct:.2f}% accurate to target)")

            # Report Brute Force proximity if it couldn't reach the exact target
            if bf_sum != T:
                print_proximity_report("Brute Force (Optimal Reference)", bf_sum, bf_diff)

            # Report Heuristic proximity if it missed the exact target
            if heur_sum != T:
                print_proximity_report("Heuristic", heur_sum, heur_diff)

            # THE SANITY CHECK CRITERIA FOR OPTIMIZATION: 
            # The heuristic cannot possibly find a subset closer to the target than the exhaustive search can.
            if heur_diff < bf_diff:
                raise ValueError(
                    f"Level 2 Logical Paradox: Heuristic managed to find a closer sum ({heur_sum}) "
                    f"than the Exhaustive Brute Force algorithm ({bf_sum}). Check your Brute Force logic!"
                )

            # If all structural and comparative checks pass for the instance
            print(f"  ✔ PASSED: Both algorithm results validated cleanly.\n")
            success_count += 1
            
        except ValueError as e:
            print(f"  ❌ VALIDATION ERROR: {e}\n")
        except IndexError:
            print(f"  ❌ PARSING ERROR: Unexpected end of file layout at instance {case_num}.\n")
            break
        except Exception as e:
            print(f"  💥 RUNTIME EXCEPTION: {e}\n")

    print(f"====================================================")
    print(f" TESTING COMPLETE: Successfully verified {success_count}/{num_cases} instances.")
    print(f"====================================================")

In [7]:
run_all_file_verification("../initial_general_tests.txt")

 STARTING ALL-INSTANCE EXPERIMENTAL VERIFICATION
 Processing 15 total instances from file

[Instance 1/15] Target T = 82 | Multiset Size = 3
  ⚠️  Brute Force (Optimal Reference) did not find exact target match.
      ↳ Sub-optimal Sum: 80 (Target: 82)
      ↳ Closeness: Off by 2 units (97.56% accurate to target)
  ⚠️  Heuristic did not find exact target match.
      ↳ Sub-optimal Sum: 80 (Target: 82)
      ↳ Closeness: Off by 2 units (97.56% accurate to target)
  ✔ PASSED: Both algorithm results validated cleanly.

[Instance 2/15] Target T = 58 | Multiset Size = 15
  ✔ PASSED: Both algorithm results validated cleanly.

[Instance 3/15] Target T = 191 | Multiset Size = 5
  ⚠️  Brute Force (Optimal Reference) did not find exact target match.
      ↳ Sub-optimal Sum: 181 (Target: 191)
      ↳ Closeness: Off by 10 units (94.76% accurate to target)
  ⚠️  Heuristic did not find exact target match.
      ↳ Sub-optimal Sum: 181 (Target: 191)
      ↳ Closeness: Off by 10 units (94.76% accurate 

### Section 6: Performance Testing:

In [27]:
import collections
from pathlib import Path
import re
import time


def verify_result(desiredSum, original_multiset, returned_subset):
    """Validates the correctness of the returned subset against constraints.

    Throws an Exception if any validation check fails.
    """
    # Check 1: Verify elements and duplicate counts using frequency mapping
    original_counts = {}
    for item in original_multiset:
        original_counts[item] = original_counts.get(item, 0) + 1

    subset_counts = {}
    for item in returned_subset:
        subset_counts[item] = subset_counts.get(item, 0) + 1

    # Validate each element in the returned subset against the original counts
    for item, count in subset_counts.items():
        if item not in original_counts:
            raise ValueError(
                f"Validation Failed: Element {item} in the returned subset "
                f"does not exist in the original multiset."
            )
        if count > original_counts[item]:
            raise ValueError(
                f"Validation Failed: Element {item} appears {count} times "
                f"in the subset, but only {original_counts[item]} times in the original multiset."
            )


def run_performance_testing(input_filename, algorithm_func, algorithm_type):
    """Reads sequential test cases from input_filename, measures execution time

    for the optimization algorithm, validates structural integrity, and logs
    data.

    Parameters:
        input_filename (str/Path): Path to the uniform test file.
        algorithm_func (callable): The optimization algorithm to run (returns
          (subset, sum_found)).
        algorithm_type (str): Either 'b' (Brute Force) or 'h' (Heuristic).
    """
    input_path = Path(input_filename)

    # 1. Generate the customized output filename based on algorithm type
    base_name = input_path.stem
    output_filename = f"result_{base_name}_{algorithm_type.lower()}"

    result_dir = Path("../results")
    file_path = result_dir / output_filename
    result_dir.mkdir(parents=True, exist_ok=True)

    # 2. Parse file input sequentially
    with open(input_path, "r") as infile:
        lines = [line.strip() for line in infile if line.strip()]

    if not lines:
        print(f"Input file '{input_path.name}' is empty.")
        return

    num_cases = int(lines[0])
    line_index = 1
    is_size_zero_file = len(lines) == num_cases + 1

    performance_records = []

    for case_num in range(1, num_cases + 1):
        if line_index >= len(lines):
            break

        # Parse uniform file rows sequentially
        T = int(lines[line_index])
        line_index += 1

        if is_size_zero_file:
            multiset = []
        else:
            if line_index < len(lines):
                multiset = list(map(int, lines[line_index].split()))
                line_index += 1
            else:
                multiset = []

        # 3. Benchmark execution
        # Swapped parameter ordering to match your updated algorithm signature: (multiset, target)
        start_time = time.perf_counter()
        result_subset, sum_found = algorithm_func(multiset, T)
        end_time = time.perf_counter()

        runtime = end_time - start_time

        # 4. Level-1 Validation: Verify chosen elements belong to the parent multiset
        verify_result(T, multiset, result_subset)

        # 5. Prep output strings
        if len(result_subset) == 0:
            subset_str = "EMPTY_SET"  # Clean fallback string if the optimal subset is empty
        else:
            subset_str = " ".join(map(str, result_subset))

        # Format: runtime sum_found target_sum subset_found (all space-separated)
        record_line = f"{runtime:.8f} {sum_found} {T} {subset_str}"
        performance_records.append(record_line)

    # 6. Write out structural logs in original chronological instance order
    with open(file_path, "w") as outfile:
        for record in performance_records:
            outfile.write(record + "\n")

    print(
        f" Successfully benchmarked {input_path.name} -> Saved to '{output_filename}'."
    )

#### For Brute Force:

In [ ]:
import re

try:
    dir_path = Path("../tests/uniform_tests")
    if dir_path.exists() and dir_path.is_dir():
        # 1. Gather and sort the uniform test files numerically
        file_list = [p for p in dir_path.iterdir() if p.is_file()]
        file_list.sort(key=lambda p: [int(x) for x in re.findall(r'\d+', p.name)] if re.findall(r'\d+', p.name) else [0])
        
        print("====================================================")
        # Assumes your brute force optimization function is named 'ExhaustiveAlgorithm'
        print(f" STARTING BENCHMARKS FOR: {ExhaustiveAlgorithm.__name__}")
        print("====================================================\n")
        
        # 2. Loop through and run performance tests
        for path in file_list:
            # 'b' tells the function to append '_b' to the resulting file name
            run_performance_testing(path, ExhaustiveAlgorithm, algorithm_type='b')
            
        print("\n✔ All Brute Force benchmarks completed successfully.")
    else:
        print("Error: '../tests/uniform_tests' directory not found.")
            
except ValueError as e:
    print(f"\n❌ [BENCHMARK PAUSED] Validation Error: {e}")
except Exception as e:
    print(f"\n💥 [BENCHMARK CRASHED] Runtime Exception: {e}")

 STARTING BENCHMARKS FOR: ExhaustiveAlgorithm

 Successfully benchmarked uniform_test_size0.txt -> Saved to 'result_uniform_test_size0_b'.
 Successfully benchmarked uniform_test_size1.txt -> Saved to 'result_uniform_test_size1_b'.
 Successfully benchmarked uniform_test_size2.txt -> Saved to 'result_uniform_test_size2_b'.
 Successfully benchmarked uniform_test_size3.txt -> Saved to 'result_uniform_test_size3_b'.
 Successfully benchmarked uniform_test_size4.txt -> Saved to 'result_uniform_test_size4_b'.
 Successfully benchmarked uniform_test_size5.txt -> Saved to 'result_uniform_test_size5_b'.
 Successfully benchmarked uniform_test_size6.txt -> Saved to 'result_uniform_test_size6_b'.
 Successfully benchmarked uniform_test_size7.txt -> Saved to 'result_uniform_test_size7_b'.
 Successfully benchmarked uniform_test_size8.txt -> Saved to 'result_uniform_test_size8_b'.
 Successfully benchmarked uniform_test_size9.txt -> Saved to 'result_uniform_test_size9_b'.
 Successfully benchmarked uniform

#### For Heuristic Algorithm:

In [15]:
import re

try:
    dir_path = Path("../tests/uniform_tests")
    if dir_path.exists() and dir_path.is_dir():
        # 1. Gather and sort the uniform test files numerically
        file_list = [p for p in dir_path.iterdir() if p.is_file()]
        file_list.sort(key=lambda p: [int(x) for x in re.findall(r'\d+', p.name)] if re.findall(r'\d+', p.name) else [0])
        
        print("====================================================")
        # Change 'HeuristicAlgorithm' to match the exact name of your heuristic function
        print(f" STARTING BENCHMARKS FOR: {rgli_subset_sum.__name__}")
        print("====================================================\n")
        
        # 2. Loop through and run performance tests
        for path in file_list:
            # 'h' tells the function to append '_h' to the resulting file name
            run_performance_testing(path, rgli_subset_sum, algorithm_type='h')
            
        print("\n✔ All Heuristic benchmarks completed successfully.")
    else:
        print("Error: '../tests' directory not found.")
            
except ValueError as e:
    print(f"\n❌ [BENCHMARK PAUSED] Validation Error: {e}")
except Exception as e:
    print(f"\n💥 [BENCHMARK CRASHED] Runtime Exception: {e}")

 STARTING BENCHMARKS FOR: rgli_subset_sum

 Successfully benchmarked uniform_test_size0.txt -> Saved to 'result_uniform_test_size0_h'.
 Successfully benchmarked uniform_test_size1.txt -> Saved to 'result_uniform_test_size1_h'.
 Successfully benchmarked uniform_test_size2.txt -> Saved to 'result_uniform_test_size2_h'.
 Successfully benchmarked uniform_test_size3.txt -> Saved to 'result_uniform_test_size3_h'.
 Successfully benchmarked uniform_test_size4.txt -> Saved to 'result_uniform_test_size4_h'.
 Successfully benchmarked uniform_test_size5.txt -> Saved to 'result_uniform_test_size5_h'.
 Successfully benchmarked uniform_test_size6.txt -> Saved to 'result_uniform_test_size6_h'.
 Successfully benchmarked uniform_test_size7.txt -> Saved to 'result_uniform_test_size7_h'.
 Successfully benchmarked uniform_test_size8.txt -> Saved to 'result_uniform_test_size8_h'.
 Successfully benchmarked uniform_test_size9.txt -> Saved to 'result_uniform_test_size9_h'.
 Successfully benchmarked uniform_tes